### IMS pipeline: extract → transform → load

| stage | class | what it does |
|---|---|---|
| extract | `DataExtractor` | parquet → `ChannelData`, drops rig-shutdown snapshots, attaches labels |
| transform | `DataTransformer` | chronological train/val/test split, z-score with **train** statistics |
| load | `DataLoader` | cuts each snapshot into windows, wraps them in a torch `DataLoader` |

The output is what a model trains on: batches of fixed-length windows of raw
vibration, each carrying its snapshot index so window-level scores can be aggregated
back to snapshot level at evaluation time.

In [ ]:
import os

os.chdir('..')
os.getcwd()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from loguru import logger

from phm_101.data_pipeline.data_extractor import DataExtractor
from phm_101.data_pipeline.data_loader import DataLoader
from phm_101.data_pipeline.data_transformer import DataTransformer
from phm_101.utils.ims import CHANNELS

logger.remove()

CHANNEL = 't2b1'  # outer race failure, the clearest degradation ramp
WINDOW_SIZE = 2048  # ~3.4 shaft revolutions at 2000 rpm / 20 kHz
HOP = 2048  # no overlap: 10 windows tile a snapshot exactly
BATCH_SIZE = 64

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} on {device}')

### 1. extract

In [ ]:
extractor = DataExtractor()
data = extractor.load(CHANNEL)

print(f'{len(data)} snapshots of {data.signals.shape[1]} samples, {data.signals.dtype}')
print(f'{data.timestamps[0]}  ->  {data.timestamps[-1]}')
print(f'{int(data.labels.sum())} faulty ({100 * data.labels.mean():.1f}%)')
print(f'fault onset: {extractor.fault_onset[CHANNEL]}')

### 2. transform

Train and val must be healthy only — that is the whole premise of training a model of
"normal" and flagging departures from it.

In [ ]:
transformer = DataTransformer()
split = transformer.train_test_split(data)
transformer.fit(split.train)  # statistics from train only

parts = {
    'train': transformer.transform(split.train),
    'val': transformer.transform(split.val),
    'test': transformer.transform(split.test),
}

pd.DataFrame([
    {
        'part': name,
        'snapshots': len(p),
        'faulty': int(p.labels.sum()),
        'from': p.timestamps[0],
        'to': p.timestamps[-1],
        'mean': p.signals.mean(),
        'std': p.signals.std(),
    }
    for name, p in parts.items()
])

Train comes out at mean 0 / std 1 by construction. Val should land close to it — if it
does not, the split has caught degradation that was assumed healthy. Test drifting away
is the fault.

### 3. load

In [ ]:
loader = DataLoader(window_size=WINDOW_SIZE, hop=HOP, batch_size=BATCH_SIZE)

loaders = {
    name: loader.get_dataloader(part, train=(name == 'train'))
    for name, part in parts.items()
}

windows, labels, snapshots = next(iter(loaders['train']))
print(f'batch   {tuple(windows.shape)}  {windows.dtype}')
print(f'labels  {tuple(labels.shape)}  {labels.dtype}')
print(f'snapshot ids {tuple(snapshots.shape)}  {snapshots.dtype}')
print()
for name, dl in loaders.items():
    print(f'{name:5s} {len(dl.dataset):7d} windows in {len(dl):5d} batches')

The dataset already yields `(1, window_size)` per item, matching the PyTorch
tutorial convention where FashionMNIST samples are `(1, 28, 28)`. So a batch
arrives as `(batch, 1, window_size)` — exactly what `Conv1d` expects, with no
reshaping in the training loop.

### 4. does the windowing preserve the signal?

Windows are cut *inside* a snapshot and must never span two of them, otherwise a window
could straddle a 10-minute gap. With `hop == window_size` the windows tile a snapshot
exactly, so concatenating them must reproduce it.

In [ ]:
dataset = loaders['train'].dataset
n_windows = dataset.n_windows

snapshot = 7
rebuilt = np.concatenate([
    dataset[snapshot * n_windows + w][0].squeeze(0).numpy() for w in range(n_windows)
])
original = parts['train'].signals[snapshot]

print(f'{n_windows} windows x {WINDOW_SIZE} = {n_windows * WINDOW_SIZE} samples')
print(f'snapshot length                  = {original.size} samples')
print(f'identical: {np.array_equal(rebuilt, original)}')

In [ ]:
# evaluation order must be deterministic, or snapshot ids cannot be aggregated
ids = torch.cat([s for _, _, s in loaders['test']])
print('snapshot ids non-decreasing:', bool((ids[1:] >= ids[:-1]).all()))
print('every snapshot windowed equally:', bool((torch.bincount(ids) == n_windows).all()))

### 5. what the model will actually see

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 5), sharey=True)

for row, name in enumerate(['train', 'val']):
    ds = loaders[name].dataset
    for col, i in enumerate([0, len(ds) // 2, len(ds) - 1]):
        axes[row, col].plot(ds[i][0].squeeze(0).numpy(), lw=0.4)
        axes[row, col].set_title(f'{name} window {i}', fontsize=9)

fig.suptitle('healthy windows (z-scored)')
fig.tight_layout()

fig, axes = plt.subplots(1, 3, figsize=(14, 2.6), sharey=True)
ds = loaders['test'].dataset
for col, i in enumerate([0, len(ds) // 2, len(ds) - 1]):
    axes[col].plot(ds[i][0].squeeze(0).numpy(), lw=0.4, color='crimson')
    axes[col].set_title(f'test window {i} (label {ds[i][1]})', fontsize=9)

fig.suptitle('test windows — the last ones are the failure')
fig.tight_layout()

### 6. one dry-run epoch

No model yet — just push every batch to the device to confirm the loader keeps up and
nothing blows up on dtype or shape.

In [ ]:
import time

start = time.perf_counter()
n_windows_seen = 0

for windows, labels, snapshots in loaders['train']:
    windows = windows.to(device)  # (batch, 1, window_size)
    n_windows_seen += windows.shape[0]

elapsed = time.perf_counter() - start
print(f'{n_windows_seen} windows in {elapsed:.2f}s ({n_windows_seen / elapsed:.0f} windows/s)')
print(f'last batch on device: {tuple(windows.shape)} {windows.device}')

### 7. the whole rig

Same pipeline for all sixteen channels. Each transformer is re-fitted per channel —
bearings sit at different baseline vibration levels, so one shared scale would be wrong.

In [ ]:
rows = []

for channel in [c for channels in CHANNELS.values() for c in channels]:
    d = extractor.load(channel)
    t = DataTransformer()
    s = t.train_test_split(d)
    t.fit(s.train)

    rows.append({
        'channel': channel,
        'failed': bool(d.labels.sum()),
        'train_windows': len(loader.get_dataloader(s.train, train=True).dataset),
        'val_windows': len(loader.get_dataloader(s.val, train=False).dataset),
        'test_windows': len(loader.get_dataloader(s.test, train=False).dataset),
        'test_faulty': int(s.test.labels.sum()),
        'fitted_std': t.std,
        'leakage': int(s.train.labels.sum() + s.val.labels.sum()),
    })
    del d, s

summary = pd.DataFrame(rows)
print('faulty snapshots leaking into train/val:', summary['leakage'].sum())
summary.round(4)

### notes for training

- **`num_workers` is a trap here.** The dataset holds a `sliding_window_view`, which is
  a zero-copy view — but pickling it (which `num_workers > 0` does on Windows, once per
  worker) materialises it. At `hop = 2048` that is 37 MB per worker, harmless; at
  `hop = 64` it is 1 GB, and at `hop = 1` it is 67 GB. Keep `num_workers=0` unless the
  hop is large.
- **`hop` must divide the snapshot cleanly** or the tail is silently dropped:
  `window_size=3000, hop=3000` covers only 18000 of 20480 samples. 2048 and 1024 tile
  exactly.
- **Fit one transformer per channel**, as above — the fitted std ranges from 0.053 to
  0.113 across the sixteen channels.